# Task 4 – Teil 3b: Spannungsregelung

**Ausgangspunkt:** das ausgelegte Netz aus `Task4-3_Ortsnetz_Auslegung.ipynb` (CIGRE-MV ohne DER, CIGRE-LV-Ortsnetz an Bus 12 mit pandapower-Standardtrafos und verstärkter Freileitung im Gewerbenetz). Dieses Notebook setzt voraus, dass Teil 3 einmal ausgeführt wurde (Simulationen `w4_t3_ausgelegt` und `w4_t3_000` … `w4_t3_111`).

**Regelkonzept:** Die Regelung wird getrennt:

- **NS-Netz:** automatische Regelung über die Stufenschalter der Ortsnetztransformatoren (regelbarer Ortsnetztransformator, rONT).
- **MS-Netz:** Backup-Plan über die Trennschalter S1–S3. Das Schließen der Schalter, vor allem von S1, ist ein starker Eingriff und nur für den Notfall gedacht. Trotzdem ist ein Regelplan sinnvoll: Tritt eine unvorhergesehene Spannungs- oder Leistungsspitze auf, liegt bereits ein Plan vor, wie dagegen gearbeitet wird. Im Normalbetrieb soll der Backup-Plan dagegen **nie ausgelöst** werden; das Netz ist so ausgelegt (Teil 2 und Teil 3), dass es ohne Kupplung der Abgänge auskommt und bei Bedarf beide Transformatoren noch als Reserve zur Verfügung stehen.

**Aufbau:**

1. Bewertung möglicher Regelungsmethoden
2. Analyse vor der Regelung: Wirkung der Schalter je MV-Bus, Regelbedarf im Ortsnetz
3. Regelkonzept: rONT-Regler und Backup-Plan
4. Kontrollierte Untersuchung ausgewählter Stufenstellungen
5. Zeitreihe ungeregelt und geregelt
6. Test des Backup-Plans bei Lastspitzen
7. Regelung bei allen Schalterstellungen S1–S3
8. Fazit

## Importe

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandapower as pp

import simulation as sim
import szenarien as sz
import lvnetz
import regelung as rg
import netplan
import plots
from elements import Diagram
from profiles import Month

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=plt.cm.tab20.colors)

sim.ONLY_RUN_IF_CHANGED = True

V_MIN, V_MAX = 0.90, 1.10                 # Arbeitsgrenzen
V_PLAN_MIN, V_PLAN_MAX = 0.95, 1.05       # planerisches Spannungsband (Teil 3)
PLAN_MARKER = [("orange", V_PLAN_MIN, "Planungsband 0,95 pu", "--"), ("orange", V_PLAN_MAX, "Planungsband 1,05 pu", "--")]

ausgelegt = sim.loadSimulation("w4_t3_ausgelegt")        # aus Task4-3 (ausgelegtes Netz, alle Trennschalter offen)
ungeregelt = {k: sim.loadSimulation(f"w4_t3_{k}") for k in sz.KEYS}   # Zeitreihen aller Schalterstellungen ohne Regelung
g = lvnetz.bus_groups(ausgelegt.net)
TEILNETZE = {x: [b for b in g["NS"] if b.startswith(f"LV Bus {x}")] for x in "RIC"}
TRAFO = {x: f"LV Trafo {x}0-{x}1" for x in "RIC"}

def band_stunden(s):
    """Stunden außerhalb des Planungsbands und der Arbeitsgrenzen, getrennt nach MS und NS."""
    vm = s.results.vm_pu[s.results.converged]
    out = {}
    for lvl, cols in (("MS", g["MS"]), ("NS", g["NS"])):
        lo, hi = vm[cols].min(axis=1), vm[cols].max(axis=1)
        out[f"Stunden < {V_PLAN_MIN} {lvl}"] = int((lo < V_PLAN_MIN).sum())
        out[f"Stunden > {V_PLAN_MAX} {lvl}"] = int((hi > V_PLAN_MAX).sum())
        out[f"Stunden < {V_MIN} {lvl}"] = int((lo < V_MIN).sum())
        out[f"Stunden > {V_MAX} {lvl}"] = int((hi > V_MAX).sum())
        out[f"U min {lvl} [pu]"] = lo.min()
        out[f"U max {lvl} [pu]"] = hi.max()
    return pd.Series(out)

## 1 Bewertung möglicher Regelungsmethoden

Betrachtet wurden drei Ansätze für die Regelung:

| | 1. Schwellwertregelung | 2. Fahrplan aus historischen Werten | 3. Prognose mit neuronalem Netz |
|---|---|---|---|
| Prinzip | reagiert auf den Messwert, wenn eine Grenze überschritten wird | Stufenstellung nach Zeitplan, abgeleitet z. B. aus dem Vorjahr | Stufenstellung nach Vorhersage des Verbrauchs der nächsten Stunde |
| Aufwand in der Simulation | gering bis mittel | gering (Stufen je Monat sind in `runSimulation(tap=...)` schon möglich) | hoch (Trainingsdaten, Merkmale, Validierung) |
| Stand der Technik | ja, so arbeiten Spannungsregler von Stufenschaltern [19, S. 714], [20], [21] | als Ergänzung | Forschung, Smart Grid |
| Schwäche | – (siehe unten) | reagiert nicht auf Abweichungen; nach dem Arbeitsauftrag keine automatische Regelung | nur ein synthetisches Jahr als Datenbasis |

**Zu 1:** Der befürchtete Nachteil, dass mindestens eine Stunde über dem Schwellwert liegt, entsteht nur durch die Stundenauflösung der Simulation. Ein realer Stufenschalter reagiert nach einer Verzögerungszeit im Bereich von Sekunden bis Minuten; der ECOTAP VPD schafft bis zu 20 Stufenschaltungen pro Minute [21]. In der Simulation wird das abgebildet, indem nach einem Eingriff **dieselbe Stunde** erneut gerechnet wird (wie bei den Controllern von pandapower). Damit bleibt keine Stunde verletzt, solange der Stellbereich reicht.

**Zu 2:** Die BDEW-Standardlastprofile sind in jedem Jahr gleich. Ein Fahrplan aus dem „Vorjahr“ wäre deshalb automatisch perfekt und damit nicht aussagekräftig. Außerdem verlangt der Arbeitsauftrag ausdrücklich eine automatische Regelung; eine fest vorgegebene Stufenstellung genügt nicht.

**Zu 3:** Mit nur einem synthetischen Jahr ist eine Verbrauchsprognose nahezu trivial und kaum belastbar. Der Nutzen gegenüber 1 ist gering, weil der Stufenschalter ohnehin innerhalb der Stunde reagiert. Die Idee passt aber in den Ausblick (Teil 9), z. B. als Prognose für Speicher und steuerbare Lasten.

**Entscheidung:** Methode 1 mit Totband, Hysterese, Stellgrenzen und Zählung der Schaltvorgänge. Methoden 2 und 3 werden nicht umgesetzt; 3 wird im Ausblick aufgegriffen.

## 2 Analyse vor der Regelung

### 2.1 Wirkung der Trennschalter auf die MV-Busse

Mittlere Änderung der Spannung je MV-Bus gegenüber „alle offen“ in den 20 Stunden mit der niedrigsten MV-Spannung (Zeitreihen aus Teil 3, ohne Regelung), in Prozentpunkten:

In [ ]:
base = ungeregelt["000"].results.vm_pu[g["MS"]]
schwach_h = base.min(axis=1).nsmallest(20).index
wirkung = pd.DataFrame({sz.label(k, kurz=True): ((s.results.vm_pu[g["MS"]] - base).loc[schwach_h].mean() * 100)
                        for k, s in ungeregelt.items()})
plots.table(wirkung, decimals=2, cmap="RdYlGn", vmin=-2.5, vmax=2.5)

Anzahl Stunden je MV-Bus unter dem Planungsband (0,95 pu) in jeder Schalterstellung:

In [ ]:
stunden = pd.DataFrame({sz.label(k, kurz=True): (s.results.vm_pu[g["MS"]] < V_PLAN_MIN).sum() for k, s in ungeregelt.items()})
stunden[stunden.sum(axis=1) > 0]

### 2.2 Regelbedarf im Ortsnetz

Je Teilnetz: niedrigste Spannung an der NS-Sammelschiene (Trafo-Unterspannungsseite) und am schwächsten Bus, Stunden unter 0,95 pu und die Zahl der Stunden, in denen eine bzw. zwei Stufen (je 2,5 %) nötig wären:

In [ ]:
vm0 = ungeregelt["000"].results.vm_pu
bedarf = {}
for x, busse in TEILNETZE.items():
    lo, hi = vm0[busse].min(axis=1), vm0[busse].max(axis=1)
    bedarf[f"Teilnetz {x}"] = {
        "Sammelschiene min [pu]": vm0[f"LV Bus {x}1"].min(),
        "schwächster Bus": vm0[busse].min().idxmin(),
        "schwächster Bus min [pu]": lo.min(),
        "Stunden < 0,95": int((lo < 0.95).sum()),
        "1 Stufe nötig": int(((lo < 0.95) & (lo >= 0.925)).sum()),
        "2 Stufen nötig": int((lo < 0.925).sum()),
        "U max [pu]": hi.max(),
    }
pd.DataFrame(bedarf).T

In [ ]:
vergleich_messpunkt = ["LV Bus R1", "LV Bus R15", "LV Bus C1", "LV Bus C12"]
ungeregelt["000"].createDiagram(Diagram.VOLTAGE, vergleich_messpunkt, months=Month.JANUARY, markers=PLAN_MARKER,
                                title="Sammelschiene (R1, C1) und Strangende (R15, C12) ohne Regelung")

**Folgerungen aus der Analyse:**

- **S2 allein wirkt praktisch nicht.** Bus 6 und Bus 7 haben fast dieselbe Spannung, über die geschlossene Masche fließt kaum Leistung.
- **S3 hebt Bus 7 bis Bus 11 um 0,15 bis 0,3 Prozentpunkte** an, senkt aber Bus 4 bis Bus 6 leicht (Bus 6 fällt dadurch in 3 Stunden unter 0,95 pu). **S2 und S3 zusammen** beseitigen die 8 Stunden unter dem Planungsband in Abgang 1 vollständig.
- **S1 hebt Abgang 1 um gut 2 Prozentpunkte**, senkt aber Abgang 2: Bus 13 um 1,3 und Bus 14 um 2,0 Prozentpunkte, Bus 12 (Anschluss des Ortsnetzes) um 0,2 Prozentpunkte. S1 hilft also dem MV-Netz, schadet aber dem Ortsnetz.
- **Größenordnung:** S2/S3 wirken mit weniger als 0,5 Prozentpunkten; sie können keine Abweichung von mehreren Prozent ausgleichen. Deshalb sind sie Stufe 1 des Backup-Plans für kleine Unterschreitungen in Abgang 1, S1 ist Stufe 2 für den Notfall.
- **Ortsnetz:** An den NS-Sammelschienen fällt die Spannung nie unter 0,984 pu. Die Verletzungen des Planungsbands treten nur an den Strangenden auf (LV Bus R15 im Wohngebiet, LV Bus C12 im Gewerbe), abends (und am Wochenende mittags) im Wohngebiet und werktags tagsüber im Gewerbe. Ein Regler, der nur die Sammelschiene misst, würde nie eingreifen. Deshalb misst der Regler die niedrigste und höchste Spannung im ganzen Teilnetz.
- **Regelbedarf:** Im Wohngebiet 195 Stunden, im Gewerbe 255 Stunden; fast immer reicht eine Stufe (2,5 %), nur in 2 Stunden wären zwei Stufen nötig. Das Industrienetz liegt immer im Band.

## 3 Regelkonzept

### 3.1 NS-Netz: regelbarer Ortsnetztransformator

| Merkmal | Festlegung | Begründung |
|---|---|---|
| geregelte Trafos | alle drei Ortsnetztrafos, jeder mit eigenem Regler | jeder Trafo wirkt nur auf sein eigenes Teilnetz; die Teilnetze haben unterschiedliche Lastgänge (Haushalt abends, Gewerbe vormittags) |
| wirksam für | nur die NS-Busse des eigenen Teilnetzes | die US-Spannung eines Trafos ändert die Spannungen im MV-Netz und in den anderen Teilnetzen praktisch nicht |
| Stufenseite | Oberspannung (`tap_side = "hv"`) | Regelwicklung üblicherweise auf der OS-Seite [19, S. 421]; Standardtyp |
| Stellbereich | −2 … +2 Stufen zu je 2,5 % | Standardtyp; entspricht dem rONT-Beispiel ±2 × 2,5 % [20] |
| Messgröße | niedrigste und höchste Spannung aller NS-Busse des Teilnetzes | entspricht Sensoren an den Strangenden; an der Sammelschiene allein wäre keine Verletzung zu sehen (Abschnitt 2.2) |
| Sollband | 0,95 … 1,05 pu (Planungsband) | Teil 3 |
| Regelgesetz | Abweichung bis 2,5 % unter bzw. über dem Band → 1 Stufe; bis 5 % → 2 Stufen | Stufen entsprechen der Abweichung |
| Stellgrenze | Anheben nur, wenn U_max danach ≤ 1,05 pu bleibt; Absenken nur, wenn U_min ≥ 0,95 pu bleibt | Behebung einer Verletzung darf keine neue erzeugen |
| Rückstellung / Hysterese | eine Stufe zurück Richtung Neutralstellung nur, wenn danach beide Grenzen mit 0,5 % Abstand im Band liegen; Wirkung je Stufe vorsichtig mit 1,2 × 2,5 % geschätzt | verhindert Hin- und Herschalten |
| Verzögerung | in der Simulation: Eingriff innerhalb derselben Stunde (Stunde wird neu gerechnet); real: Verzögerungszeit im Bereich von Sekunden bis Minuten [21] | Stundenauflösung |
| Vorzeichen | `tap_pos = −1` hebt die US-Spannung um eine Stufe an | geprüft mit pandapower |

Der Regler berücksichtigt Unter- und Überspannung. Ohne Erzeuger tritt Überspannung bisher nicht auf; mit den Erzeugungsszenarien in Teil 4 und bei Leistungsflussumkehr wirkt dieselbe Logik nach unten.

### 3.2 MS-Netz: Backup-Plan mit den Trennschaltern

| Stufe | Maßnahme | Auslösung (niedrigste MV-Spannung) | Rücknahme | Begründung |
|---|---|---|---|---|
| 1 | S2 und S3 schließen | < 0,94 pu | ≥ 0,95 pu | schließt nur die Maschen innerhalb von Abgang 1; beide Transformatoren bleiben getrennt; hebt Bus 7 bis 11 an (Abschnitt 2.1). S2 allein wirkt praktisch nicht. |
| 2 | S1 schließen | < 0,92 pu | ≥ 0,96 pu | kuppelt beide Abgänge, hebt Abgang 1 um gut 2 Prozentpunkte, senkt aber Abgang 2 (Bus 12 bis 14) und macht das Netz von beiden Transformatoren abhängig; daher nur als Notfallmaßnahme |

- Die Auslöseschwellen liegen bewusst **unter** dem Planungsband: Stufe 1 bei mehr als 1 Prozentpunkt Unterschreitung, Stufe 2, wenn nur noch 2 Prozentpunkte Abstand zur Arbeitsgrenze 0,90 pu bleiben. Der Backup-Plan ist eine Notfallmaßnahme und soll im Normalbetrieb nicht eingreifen.
- Die Rücknahmeschwellen liegen um die Wirkung der Stufe plus Reserve höher, damit nach dem Öffnen nicht sofort wieder geschlossen wird.
- Je Prüfung wird höchstens eine Stufe zu- oder abgeschaltet; Rücknahme in umgekehrter Reihenfolge.

### 3.3 Umsetzung

`regelung.py` enthält die Klassen `TapRegler` (rONT) und `SchaltPlan` (Backup-Plan). Sie werden an `runSimulation(controllers=[...])` übergeben. Nach jedem Lastfluss prüft jeder Regler das Ergebnis; greift einer ein, wird dieselbe Stunde neu gerechnet, bis keiner mehr eingreift (höchstens 10-mal). Stufenstellungen und Schalterzustände werden in die nächste Stunde übernommen und je Stunde gespeichert (`results.tap_pos`, `results.switch_closed`).

In [ ]:
def lv_regler():
    return [rg.TapRegler(TRAFO[x], TEILNETZE[x], u_min=V_PLAN_MIN, u_max=V_PLAN_MAX) for x in "RIC"]

def backup_plan():
    return rg.SchaltPlan(g["MS"], [(["S2", "S3"], 0.94, 0.95), (["S1"], 0.92, 0.96)])

for r in lv_regler():
    print(r.trafo, "→", len(r.busse), "überwachte NS-Busse")

## 4 Kontrollierte Untersuchung ausgewählter Stufenstellungen

Bevor automatisch geregelt wird, wird kontrolliert untersucht, wie feste Stufenstellungen (−2 … +2, an allen drei Ortsnetztrafos gleich) die Spannungen verändern. Zwei statische Betriebspunkte:

- **Spitzenlast:** alle Lasten mit Bemessungsleistung
- **Schwachlast:** alle Lasten mit 25 % der Bemessungsleistung (etwa die Nachtlast)

In [ ]:
def stufen_test(faktor):
    rows = {}
    for tap in range(-2, 3):
        net = copy.deepcopy(ausgelegt.net)
        net.load[["p_mw", "q_mvar"]] *= faktor
        for x in "RIC":
            net.trafo.loc[net.trafo.name == TRAFO[x], "tap_pos"] = tap
        pp.runpp(net)
        vm = pd.Series(net.res_bus.vm_pu.to_numpy(), index=net.bus.name)
        row = {"U min MS [pu]": vm[g["MS"]].min(), "U max MS [pu]": vm[g["MS"]].max()}
        for x, busse in TEILNETZE.items():
            row[f"U min {x} [pu]"] = vm[busse].min()
            row[f"U max {x} [pu]"] = vm[busse].max()
        rows[tap] = row
    out = pd.DataFrame(rows).T
    out.index.name = "Stufe (tap_pos)"
    return out

spitze = stufen_test(1.0)
schwach = stufen_test(0.25)
display(plots.table(spitze, decimals=3, cmap="RdYlGn", vmin=0.90, vmax=1.03))
plots.table(schwach, decimals=3, cmap="RdYlGn_r", vmin=0.97, vmax=1.10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, (titel, df) in zip(axes, (("Spitzenlast", spitze), ("Schwachlast (25 %)", schwach))):
    for x, farbe in zip("RIC", ("tab:blue", "tab:green", "tab:orange")):
        ax.fill_between(df.index, df[f"U min {x} [pu]"], df[f"U max {x} [pu]"], alpha=0.25, color=farbe, label=f"Teilnetz {x}")
        ax.plot(df.index, df[f"U min {x} [pu]"], "o-", color=farbe)
        ax.plot(df.index, df[f"U max {x} [pu]"], "s--", color=farbe)
    ax.plot(df.index, df["U min MS [pu]"], "k^-", label="MS min")
    for v, c, ls in ((V_MIN, "red", "--"), (V_MAX, "red", "--"), (V_PLAN_MIN, "orange", ":"), (V_PLAN_MAX, "orange", ":")):
        ax.axhline(v, color=c, ls=ls, lw=1)
    ax.set_xticks(range(-2, 3))
    ax.set_xlabel("Stufe tap_pos (−: US-Spannung höher)")
    ax.set_title(titel)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Spannung [pu] (Fläche: min … max im Teilnetz)")
axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

**Ergebnis der kontrollierten Untersuchung:**

- Eine Stufe verändert die NS-Spannungen im ganzen Teilnetz um etwa 2,7 bis 2,8 Prozentpunkte (etwas mehr als die nominellen 2,5 %, weil auch der Spannungsfall im Trafo und in den Leitungen mit der höheren Spannung sinkt). Die MS-Spannungen ändern sich nicht.
- **Spitzenlast:** Ohne Stufe liegen Wohngebiet und Gewerbe bei 0,913 bzw. 0,917 pu. Eine Stufe (−1) reicht mit 0,941 bzw. 0,945 pu noch nicht ganz, zwei Stufen (−2) heben sie auf 0,971 bzw. 0,974 pu.
- **Schwachlast:** Mit zwei Stufen (−2) würden die NS-Spannungen auf bis zu 1,073 pu steigen und das Planungsband überschreiten; mit einer Stufe (−1) bleiben sie bei 1,045 pu im Band.
- **Folgerung:** Keine feste Stufenstellung passt für beide Betriebspunkte. Eine Stellung, die die Spitzenlast abdeckt, erzeugt bei Schwachlast Überspannung. Deshalb ist eine automatische Regelung nötig, die die Stufe mit der Last nachführt.

## 5 Zeitreihe ungeregelt und geregelt

Gleiche Randbedingungen wie die ungeregelte Zeitreihe `w4_t3_000` (alle Trennschalter offen); zusätzlich die drei rONT-Regler und der Backup-Plan.

In [ ]:
geregelt = ausgelegt.copy("w4_t3_geregelt", overwrite=True)
geregelt.runSimulation(controllers=lv_regler() + [backup_plan()])

In [ ]:
vgl = pd.DataFrame({"ungeregelt": band_stunden(ungeregelt["000"]), "geregelt": band_stunden(geregelt)})
display(plots.table(vgl.astype(float), decimals=3))
schalt = rg.schaltvorgaenge(geregelt)
display(schalt.to_frame("Anzahl in 12 Wochen"))
print("Stunden mit Reglereingriff:", int((geregelt.results.info.regler_iterationen > 0).sum()), "von", len(geregelt.results.info))

In [ ]:
taps = geregelt.results.tap_pos[[TRAFO[x] for x in "RIC"]]
haeufigkeit = taps.apply(pd.Series.value_counts).fillna(0).astype(int)
haeufigkeit.index.name = "Stufe (Stunden)"
display(haeufigkeit)

fig, ax = plt.subplots(figsize=(13, 3.2))
idx = geregelt.results.info.index[geregelt.results.info.month == "JANUARY"]
for x, farbe in zip("RIC", ("tab:blue", "tab:green", "tab:orange")):
    ax.step(idx, taps.loc[idx, TRAFO[x]], where="post", color=farbe, label=TRAFO[x])
plots.week_axis(ax, idx)
ax.set_ylabel("Stufe tap_pos")
ax.set_yticks(range(-2, 3))
ax.set_title("Stufenstellungen der Ortsnetztrafos – Januar", pad=22)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
schwaechste = [vm0[TEILNETZE[x]].min().idxmin() for x in "RC"]
sim.compare([ungeregelt["000"], geregelt], Diagram.VOLTAGE, schwaechste, months=Month.JANUARY, markers=PLAN_MARKER,
            title="Schwächste NS-Busse – ungeregelt und geregelt")
geregelt.createDiagram(Diagram.VOLTAGE, g["NS"], layout="boxplot", markers=PLAN_MARKER, title="NS-Spannungen geregelt")

In [ ]:
geregelt.createDiagram(Diagram.NETWORK, time=["min", "max_loading"], title="Geregeltes Netz")

**Ergebnis ungeregelt gegenüber geregelt (alle Trennschalter offen):**

| Kennwert | ungeregelt | geregelt |
|---|---|---|
| Stunden NS < 0,95 pu | 450 | **0** |
| niedrigste NS-Spannung | 0,923 pu | **0,950 pu** |
| höchste NS-Spannung | 1,021 pu | 1,027 pu |
| Stunden MS < 0,95 pu | 8 | 8 |
| Verletzungen der Arbeitsgrenzen | 0 | 0 |
| Stufenschritte in 12 Wochen | – | R: 92, I: 0, C: 100 |
| Schaltungen S1–S3 (Backup-Plan) | – | 0 |

- Die Regler beseitigen alle 450 Stunden unter dem Planungsband im Ortsnetz. Sie greifen in 187 der 2016 Stunden ein und nutzen nur die Stufe −1 (Wohngebiet 238 Stunden, Gewerbe 335 Stunden); der Rest der Zeit steht der Stufenschalter in Neutralstellung. Eine neue Überspannung entsteht nicht (höchstens 1,027 pu).
- Die Stufen folgen dem Tagesgang der Lasten: Im Wohngebiet wird vor allem abends (17 bis 21 Uhr) und am Wochenende auch mittags hochgeschaltet, im Gewerbe an Werktagen tagsüber (8 bis 17 Uhr), nie am Wochenende. Nachts kehren beide Regler in die Neutralstellung zurück. Hochgerechnet auf ein Jahr sind das etwa 400 bis 430 Stufenschritte je Trafo; ein ECOTAP VPD ist für 500 000 Schaltspiele ausgelegt [21].
- **Grenzen der Regelung:** Die 8 Stunden unter 0,95 pu an Bus 9 bis Bus 11 im MV-Netz (niedrigste Spannung 0,949 pu) kann der rONT nicht beheben, weil er nur auf sein Ortsnetz wirkt. Sie liegen über der Auslöseschwelle des Backup-Plans (0,94 pu); der Backup-Plan wurde wie beabsichtigt nie ausgelöst.

## 6 Test des Backup-Plans bei Lastspitzen

In der Zeitreihe greift der Backup-Plan wie beabsichtigt nicht ein. Um zu zeigen, dass er im Notfall funktioniert, wird er an statischen Lastspitzen getestet: alle Lasten mit 100 %, 110 % und 120 % ihrer Bemessungsleistung (gleichzeitig). Die Tabellen zeigen die Eingriffe Schritt für Schritt.

In [ ]:
for faktor in (1.0, 1.1, 1.2):
    net = copy.deepcopy(ausgelegt.net)
    net.load[["p_mw", "q_mvar"]] *= faktor
    print(f"Lastspitze {faktor:.0%} der Bemessungsleistung")
    display(rg.regle_statisch(net, lv_regler() + [backup_plan()]))

**Überspannung (Test der Regelrichtung nach unten):** Schwachlast (25 %) und Slack-Spannung 1,08 pu statt 1,03 pu, z. B. nach einer Spannungsanhebung im vorgelagerten Netz.

In [ ]:
net = copy.deepcopy(ausgelegt.net)
net.load[["p_mw", "q_mvar"]] *= 0.25
net.ext_grid["vm_pu"] = 1.08
rg.regle_statisch(net, lv_regler() + [backup_plan()])

**Ergebnis Backup-Plan:**

| Lastspitze | ohne Eingriff (U min MS/NS) | Eingriffe | danach (U min MS/NS) |
|---|---|---|---|
| 100 % | 0,913 pu | Stufen R −2, I −1, C −2; Stufe 1 (S2+S3) | 0,926 pu (MS), NS 0,971 pu |
| 110 % | 0,899 pu (unter Arbeitsgrenze) | Stufen; Stufe 1, dann Stufe 2 (S1) | 0,948 pu |
| 120 % | 0,885 pu (unter Arbeitsgrenze) | Stufen am Anschlag (−2); Stufe 1 und 2 | 0,939 pu |

- Bei Spitzenlast hebt der Backup-Plan zuerst die Maschen in Abgang 1 (S2+S3), bei weiter sinkender Spannung kuppelt er über S1. Bei 110 % und 120 % Last verhindern die Eingriffe eine Verletzung der Arbeitsgrenze von 0,90 pu.
- Bei 120 % stehen die Stufenschalter am Anschlag; das Planungsband wird dann nicht mehr ganz erreicht (0,939 pu), die Arbeitsgrenzen bleiben eingehalten. Für noch höhere Lasten wären weitere Maßnahmen nötig (z. B. Netzverstärkung oder Lastmanagement).
- **Überspannung:** Bei Schwachlast und 1,08 pu am Slack steigen die NS-Spannungen auf 1,070 pu. Die Regler schalten eine Stufe nach unten (+1) und bringen das Ortsnetz auf 1,028 bis 1,043 pu zurück. Die MS-Spannung von 1,074 pu kann weder der rONT noch der Backup-Plan senken; dafür wären der Umspanntransformator oder Blindleistung nötig.

## 7 Regelung bei allen Schalterstellungen S1–S3

Die drei rONT-Regler werden für jede feste Schalterstellung über 12 Wochen gerechnet (ohne Backup-Plan, weil die Schalter hier fest vorgegeben sind).

In [ ]:
geregelt_alle = sz.zeitreihe_alle(ausgelegt, prefix="w4_t3r", controllers=lv_regler())

In [ ]:
tab = pd.DataFrame({sz.label(k, kurz=True): pd.concat([band_stunden(s), rg.schaltvorgaenge(s).filter(like="Stufen")])
                    for k, s in geregelt_alle.items()})
tab_u = pd.DataFrame({sz.label(k, kurz=True): band_stunden(s) for k, s in ungeregelt.items()})
display(plots.table(tab_u.astype(float), decimals=3))   # ungeregelt
plots.table(tab.astype(float), decimals=3)              # geregelt

**Ergebnis aller Schalterstellungen:** In allen acht Stellungen liegt das Ortsnetz mit Regelung immer im Planungsband (niedrigste NS-Spannung 0,950 pu). Mit geschlossenem S1 sinkt die Spannung an Bus 12; die Regler gleichen das mit etwas mehr Stufenschritten aus (Wohngebiet 96 statt 92), die höchste NS-Spannung steigt auf 1,039 pu, bleibt aber im Band. Die verbleibenden Unterschreitungen im MV-Netz (8, 3 bzw. 9 Stunden bei „alle offen“, „S3 zu“ und „S2 zu“) hängen nur von der Schalterstellung ab, nicht von der rONT-Regelung.

## 8 Fazit

- **NS-Netz:** Drei unabhängige rONT-Regler mit Fernmessung im eigenen Teilnetz halten das Ortsnetz in allen Schalterstellungen im Planungsband 0,95 bis 1,05 pu (vorher 450 Stunden darunter). Dafür reicht fast immer eine Stufe; im Jahr sind etwa 400 bis 430 Stufenschritte je Trafo zu erwarten. Der Regler wirkt nur auf das eigene Teilnetz; der Industrietrafo muss im Normalbetrieb nicht eingreifen.
- **Messpunkt:** Entscheidend ist die Messung an den Strangenden; an der Sammelschiene wäre keine Verletzung zu sehen gewesen.
- **Feste Stufenstellung reicht nicht:** Eine Stellung, die die Spitzenlast abdeckt, erzeugt bei Schwachlast Überspannung (bis 1,073 pu); die Stufe muss automatisch nachgeführt werden.
- **MS-Netz:** Der Backup-Plan (S2+S3, dann S1) wird im Normalbetrieb nie ausgelöst, verhindert aber bei Lastspitzen von 110 % und 120 % eine Verletzung der Arbeitsgrenze. Die Kupplung über S1 bleibt eine Notfallmaßnahme.
- **Grenzen:** Der rONT wirkt nicht auf das MV-Netz (8 Stunden knapp unter 0,95 pu an Bus 9 bis 11 bleiben) und nicht auf Überspannung im MV-Netz. Die Regelung arbeitet mit Stundenwerten und ohne Totzeit; Leistungsflussumkehr durch Erzeugung wird erst mit den Szenarien in Teil 4 geprüft.

### Quellen

[15] DIN EN 50160, „Merkmale der Spannung in öffentlichen Elektrizitätsversorgungsnetzen“ (EN 50160:2010 mit Änderungen), Berlin: Beuth.

[19] A. J. Schwab, *Elektroenergiesysteme: Erzeugung, Transport, Übertragung und Verteilung elektrischer Energie*, 3. Aufl. Berlin, Heidelberg: Springer, 2012, doi: 10.1007/978-3-642-21958-0.

[20] Maschinenfabrik Reinhausen, „Regelbarer Ortsnetztransformator“, Vortrag, Lehrstuhl für Elektrische Energiesysteme, FAU Erlangen-Nürnberg. Online: https://www.ees.tf.fau.de/files/2017/12/MR_Regelbarer_Ortsnetztrafo.pdf

[21] Maschinenfabrik Reinhausen, „ECOTAP VPD (24 kV)“, Produktinformation. Online: https://www.reinhausen.com/fileadmin/downloadcenter/products/oltc/ecotap_vpd/flyer/ecotap_vpd_24kv_de.pdf